In [1]:
#input: [1,2,1,2,3]
#output: {(1,2): 2, (2,3): 1, (2,1): 1}

## lol this is a good DSA question

def get_groups(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

input = [1,2,1,2,3]
get_groups(input)

{(1, 2): 2, (2, 1): 1, (2, 3): 1}

In [6]:
# merge function, input = [1,2,1,2,3] pair (1,2) id = 99
#output = [99, 99, 3]

def merge(ids, pair, id):
    new_id = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_id.append(id)
            i += 2
        else:
            new_id.append(ids[i])
            i += 1
    return new_id

In [8]:
text = "hug uh hug uh huge"
tokens = list(text.encode("utf-8"))
print(f"original len: {len(tokens)}")

num_merges = 5
vocab_size = 256 + num_merges
merges = {}

for i in range(num_merges):
    stats = get_groups(tokens)
    best_pair = max(stats, key=stats.get)
    id = 256 + i
    print(f"Merging: {best_pair} into new token {id}")
    tokens = merge(tokens, best_pair, id)
    merges[best_pair] = id

print(f"Final length: {len(tokens)}")
print(f"Compression ratio: {len(text.encode('utf-8')) / len(tokens):.2f}X")

original len: 18
Merging: (104, 117) into new token 256
Merging: (256, 103) into new token 257
Merging: (257, 32) into new token 258
Merging: (258, 117) into new token 259
Merging: (259, 104) into new token 260
Final length: 6
Compression ratio: 3.00X


In [10]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
    
    def train(self, text, num_merges):
        ids = list(text.encode("utf-8"))

        for i in range(num_merges):
            stats = {}
            for pair in zip(ids, ids[i+1]):
                stats[pair] = stats.get(pair, 0) + 1
            best_pair = max(stats, key=stats.get)
            idx = 256 + 1
            self.merges[best_pair] = idx
            self.vocab[idx] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            newids = []
            j = 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == best_pair[0] and ids[j+1] == best_pair[1]:
                    newids.append(idx)
                    j += 2
                else:
                    newids.append(ids[j])
                    j += 1
            ids = newids
            print(f"merge {i+1}/{num_merges}: {best_pair} -> {idx}")
    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = {}
            for pair in zip(ids, ids[1:]):
                stats[pair] = stats.get(pair, 0) + 1
            
            can_merge = {p: self.merges[p] for p in stats if p in self.merges}

            if not can_merge:
                pass

            pair_to_merge = min(can_merge, key=can_merge.get)
            idx = self.merges[pair_to_merge]
            newids = []
            j = 0

            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == best_pair[0] and ids[j+1] == best_pair[1]:
                    newids.append(idx)
                    j += 2
                else:
                    newids.append(ids[j])
                    j += 1
            ids = newids
        return ids
    
    def decode(self, ids):
        tokens = b"".join(self.vocab[idx] for idx in ids)
        return tokens.decode("utf-8" ,errors="replace")